# Points on Sphere Maximizing Volume

In [ ]:
#@title Verification code

import numpy as np
import math
from scipy.spatial import ConvexHull, QhullError


def calculate_polytope_volume(
    placements_params: np.ndarray, num_points: int, dimension: int
) -> float:
  """Calculates the volume of the convex hull of points on the unit sphere.

  Points are first normalized to lie on the unit sphere, then the volume of
  their convex hull is computed.
  """
  if not isinstance(placements_params, np.ndarray):
    return -np.inf
  if placements_params.shape != (num_points, dimension):
    return -np.inf
  if np.isnan(placements_params).any():
    return -np.inf

  placements_copy = placements_params.copy()
  norms = np.linalg.norm(placements_copy, axis=1)
  if np.any(norms < 1e-9):
    return 0.0
  placements_copy /= norms[:, np.newaxis]

  try:
    hull = ConvexHull(placements_copy, qhull_options='QJ')
  except (QhullError, ValueError):
    return 0.0

  total_volume = 0.0
  try:
    d_factorial = float(math.factorial(dimension))
  except ValueError:
    return 0.0

  for simplex_indices in hull.simplices:
    facet_vertices = placements_copy[simplex_indices]
    sign, logdet = np.linalg.slogdet(facet_vertices)
    if sign != 0:
      volume_of_simplex = np.exp(logdet) / d_factorial
      total_volume += volume_of_simplex

  return total_volume

In [ ]:
#@title Initial program

import time
import numpy as np


def search_for_best_arrangement(num_points, dimension):
  """Searches for point arrangement on unit sphere maximizing polytope volume.

  Uses random mutation search: starts from a random configuration and
  iteratively mutates one point at a time, keeping improvements.

  Args:
    num_points: Number of vertices of the polyhedron.
    dimension: Dimension of the space (3 for unit sphere S^2).

  Returns:
    Array of shape (num_points, dimension) with the best arrangement found.
  """
  placed_circles_params = np.random.rand(num_points, dimension)
  best_score = calculate_polytope_volume(placed_circles_params, num_points, dimension)
  best_placements_params = placed_circles_params.copy()

  start_time = time.time()
  while time.time() - start_time < 100:
    index_to_mutate = np.random.randint(0, len(placed_circles_params))
    placed_circles_params[index_to_mutate] += np.random.uniform(
        -0.0001, 0.0001, dimension
    )
    score = calculate_polytope_volume(placed_circles_params, num_points, dimension)
    if score > best_score:
      best_score = score
      best_placements_params = placed_circles_params.copy()
    if np.random.rand() < 0.1:
      placed_circles_params = best_placements_params.copy()

  return best_placements_params

**Prompt used**

The problem of maximizing inscribed polytope volume

Act as an expert in optimization and computational geometry. Your goal is to solve the following problem: for a given dimension d, and a number of points num_points, find a configuration of num_points points on the unit sphere in d dimensions such that the volume of the convex hull (polytope) spanned by these points is as large as possible.

Your task is to produce a search function that finds the best configuration of num_points points. The points should be represented as a NumPy array of shape (num_points, dimension). Your solution will be evaluated by the following scoring function, which you have access to and do not need to implement:

def calculate_polytope_volume(
    placements_params: np.ndarray, num_points: int, dimension: int
) -> float:
  """Calculates the volume of the convex hull of a set of points.

The score is the volume of the polytope spanned by the points. Points are
  first normalized to lie on the unit sphere.

Args:
    placements_params: A NumPy array representing the coordinates of the points.
    num_points: The number of points in the arrangement.
    dimension: The number of dimensions for the space.

Returns:
    The calculated volume, which serves as the score. A higher score is better.
  """
  # --- Input validation ---
  if not isinstance(placements_params, np.ndarray):
      return -np.inf
  if placements_params.shape != (num_points, dimension):
    return -np.inf
  if np.isnan(placements_params).any():
    return -np.inf

placements_copy = placements_params.copy()
  # --- Normalization to project points onto the unit sphere ---
  norms = np.linalg.norm(placements_copy, axis=1)
  if np.any(norms < 1e-9):
    return 0.0  # Zero volume for points at the origin
  placements_copy /= norms[:, np.newaxis]

# --- Scoring: Calculate Convex Hull Volume ---
  try:
    # Use ConvexHull to find the facets of the polytope
    hull = ConvexHull(placements_copy, qhull_options='QJ')
  except (QhullError, ValueError):
    # This happens if points don't form a D-dim shape (e.g., are coplanar).
    # The volume is 0 in such degenerate cases.
    return 0.0

# The volume is calculated by summing the volumes of the pyramids
  # formed by each facet and the origin.
  total_volume = 0.0
  d_factorial = float(math.factorial(dimension))
  for simplex_indices in hull.simplices:
    facet_vertices = placements_copy[simplex_indices]
    sign, logdet = np.linalg.slogdet(facet_vertices)
    if sign != 0:
      volume_of_simplex = np.exp(logdet) / d_factorial
      total_volume += volume_of_simplex

return total_volume

This scoring function relies on scipy.spatial.ConvexHull. If the provided points are degenerate (e.g., all lie on a plane in 3D space) and cannot form a valid d-dimensional polytope, the function will correctly return a score of 0.0.

You can code up any search method you wish. You are allowed to call calculate_polytope_volume() as many times as you want. Your objective is to make the score as high as possible.

Your search function will have 100 seconds to run. After this time, it must return the best configuration of points it has found. If it fails to return within the time limit, its result will not be considered.

You can access the best construction we have found so far through the placed_points_xy variable, where x should be replaced by the dimension, and y should be replaced by the number of points. An example of how to load this is provided in the code.

## What AlphaEvolve found

Using the standard search mode, AlphaEvolve was able to quickly match the first approximately 60 results reported in Sloane's database of optimal polytopes, matching all 13 reported digits of the maximum volume. It did not manage to improve any of them.